# Extended Analysis: Perturbation vs Baseline Comparison

This notebook extends the case study analysis by comparing perturbation effects against baseline measurements, providing statistical tests and visualizations.

In [ ]:
import pandas as pd
import numpy as np
import json
from scipy import stats
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('dataset.csv')
with open('../results/baselines.json', 'r') as f:
    baselines = json.load(f)
with open('../results/evaluation_merged.json', 'r') as f:
    evaluations = json.load(f)

In [ ]:
def calculate_mi(x, y):
    """Calculate mutual information between two response arrays."""
    joint = pd.crosstab(x, y, normalize=True)
    p_x = joint.sum(axis=1)
    p_y = joint.sum(axis=0)
    mi = 0
    for i in joint.index:
        for j in joint.columns:
            if joint.loc[i, j] > 0:
                mi += joint.loc[i, j] * np.log2(joint.loc[i, j] / (p_x[i] * p_y[j]))
    return mi

def bootstrap_mi(x, y, n_bootstrap=1000):
    """Calculate MI with bootstrap standard error."""
    mis = []
    n = len(x)
    for _ in range(n_bootstrap):
        indices = np.random.choice(n, n, replace=True)
        boot_mi = calculate_mi(x.iloc[indices], y.iloc[indices])
        mis.append(boot_mi)
    return np.mean(mis), np.std(mis)

In [ ]:
PERTURBATION_TYPES = ['gender_swap', 'gender_remove', 'stylistic_uncertain', 'stylistic_colorful']
QUESTIONS = ['MANAGE', 'VISIT', 'RESOURCE']
SEEDS = [0, 1, 42]

def compute_all_metrics(df, evaluations, baselines):
    """Compute MI, shift rate, Fleiss' kappa for perturbations and baselines."""
    results = []

    for ptype in PERTURBATION_TYPES:
        for question in QUESTIONS:
            # Get original and perturbed responses
            orig_responses = df[f'{question}_original']
            pert_responses = df[f'{question}_{ptype}']
            base_responses = df[f'{question}_{ptype}_baseline']

            # MI for perturbation
            mi_pert, mi_pert_se = bootstrap_mi(orig_responses, pert_responses)

            # MI for baseline
            mi_base, mi_base_se = bootstrap_mi(orig_responses, base_responses)

            # Shift rates
            shift_pert = (orig_responses != pert_responses).mean()
            shift_base = (orig_responses != base_responses).mean()

            # Statistical tests
            # (Will need paired data for proper test)

            results.append({
                'perturbation': ptype,
                'question': question,
                'mi_perturbation': mi_pert,
                'mi_perturbation_se': mi_pert_se,
                'mi_baseline': mi_base,
                'mi_baseline_se': mi_base_se,
                'shift_rate_perturbation': shift_pert,
                'shift_rate_baseline': shift_base
            })

    return pd.DataFrame(results)

In [ ]:
def run_paired_tests(metrics_df):
    """Run paired t-test and Wilcoxon for MI comparison."""
    mi_pert = metrics_df['mi_perturbation'].values
    mi_base = metrics_df['mi_baseline'].values

    # Paired t-test
    t_stat, t_pval = stats.ttest_rel(mi_pert, mi_base)

    # Wilcoxon signed-rank
    w_stat, w_pval = stats.wilcoxon(mi_pert, mi_base)

    return {
        'paired_ttest_statistic': t_stat,
        'paired_ttest_pvalue': t_pval,
        'wilcoxon_statistic': w_stat,
        'wilcoxon_pvalue': w_pval
    }

In [ ]:
def plot_comparison(metrics_df):
    """Plot perturbation vs baseline MI comparison."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # MI comparison
    ax = axes[0]
    x = np.arange(len(metrics_df))
    width = 0.35
    ax.bar(x - width/2, metrics_df['mi_perturbation'], width,
           yerr=metrics_df['mi_perturbation_se'], label='Perturbation', color='steelblue')
    ax.bar(x + width/2, metrics_df['mi_baseline'], width,
           yerr=metrics_df['mi_baseline_se'], label='Baseline', color='coral')
    ax.set_ylabel('Mutual Information')
    ax.set_title('MI: Perturbation vs Baseline')
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r['perturbation']}\n{r['question']}"
                        for _, r in metrics_df.iterrows()], rotation=45, ha='right')
    ax.legend()

    # Shift rate comparison
    ax = axes[1]
    ax.bar(x - width/2, metrics_df['shift_rate_perturbation'], width,
           label='Perturbation', color='steelblue')
    ax.bar(x + width/2, metrics_df['shift_rate_baseline'], width,
           label='Baseline', color='coral')
    ax.set_ylabel('Shift Rate')
    ax.set_title('Shift Rate: Perturbation vs Baseline')
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r['perturbation']}\n{r['question']}"
                        for _, r in metrics_df.iterrows()], rotation=45, ha='right')
    ax.legend()

    plt.tight_layout()
    plt.savefig('../results/comparison_plot.png', dpi=150)
    plt.show()

In [ ]:
# Run the analysis
metrics_df = compute_all_metrics(df, evaluations, baselines)
print("Metrics Summary:")
print(metrics_df.to_string())
print("\n")

# Run statistical tests
test_results = run_paired_tests(metrics_df)
print("Statistical Tests:")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}")
print("\n")

# Generate visualization
plot_comparison(metrics_df)